#### Faiss
Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("speech.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size = 1000,chunk_overlap = 30)
docs = text_splitter.split_documents(documents)
docs

d:\Data Science and Gen AI Bootcamp\Udemy Material\Langchain\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct our

In [3]:
## Create the vector store
embeddings = OllamaEmbeddings(model="gemma:2b")

## FAISS is a vector store that allows us to search for similar documents based on their embeddings. We can create a FAISS index from our documents and their embeddings
## Then use it to perform similarity searches.
db = FAISS.from_documents(docs, embeddings)
db

In [ ]:
### querying 
query = "How does the speaker describe the desired outcome of the war?"

## similarity search will return the most similar document to the query based on the embeddings. 
## We can then access the page content of the most similar document.
docs = db.similarity_search(query)
docs[0].page_content

'…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [ ]:
## Alternatively, we can use the retriever interface to perform the similarity search. The retriever will return the most similar documents to the query based on the embeddings. 
## We can then access the page content of the most similar document.
retriever = db.as_retriever()
docs = retriever.invoke(query)
docs[0].page_content

'…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [6]:
## We can also get the similarity score for each document returned by the search. 
## The score is a measure of how similar the document is to the query, with higher scores indicating greater similarity.
## Uses Manhattan distance by default.
docs_and_score = db.similarity_search_with_score(query)
docs_and_score

[(Document(id='5866c05b-3306-438f-9a5b-d5acd6d4578d', metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
  np.float32(2944.6694)),
 (Document(id='23165ef5-9f45-45d8-ad1c-6565c16ead4c', metadata={'source': 'speech.txt'}, page_content='We have borne with their present government through all these bitter months because of that

In [7]:
embedding_vector = embeddings.embed_query(query)
embedding_vector

[0.31148338317871094,
 -2.143012523651123,
 0.26571419835090637,
 1.0086249113082886,
 0.607218861579895,
 0.541415810585022,
 -0.8828568458557129,
 0.07403114438056946,
 -0.09272390604019165,
 -0.7998059988021851,
 1.1228221654891968,
 -0.028041120618581772,
 -1.0696605443954468,
 1.0573655366897583,
 0.09250002354383469,
 -1.0049258470535278,
 3.065927267074585,
 1.7426280975341797,
 1.1930344104766846,
 0.7598130106925964,
 0.32262229919433594,
 -0.2863368093967438,
 0.28250110149383545,
 1.3313835859298706,
 0.2214023321866989,
 -0.3950870931148529,
 -1.3478519916534424,
 -1.289428472518921,
 -0.5422183275222778,
 -2.0764195919036865,
 -0.26619306206703186,
 -1.418036937713623,
 1.2231526374816895,
 -0.9660102128982544,
 -0.46344467997550964,
 -0.2744143307209015,
 1.8788783550262451,
 0.5613512992858887,
 0.13905370235443115,
 -0.5600343942642212,
 0.3757276237010956,
 0.4949365258216858,
 0.9321727156639099,
 -1.2976096868515015,
 -1.3897422552108765,
 0.6539187431335449,
 0.0291

In [9]:
## We can also perform a similarity search using the embedding vector directly.
## This allows us to get the similarity score for each document returned by the search.
docs_score = db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='5866c05b-3306-438f-9a5b-d5acd6d4578d', metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='23165ef5-9f45-45d8-ad1c-6565c16ead4c', metadata={'source': 'speech.txt'}, page_content='We have borne with their present government through all these bitter months because of that friendship—exercising a pat

In [10]:
## Saving And Loading
## save_local will save the FAISS index to a local directory. The directory will contain the index file and a metadata file that contains the mapping between the document ids and the original documents.
db.save_local("faiss_index")

In [11]:
## load_local will load the FAISS index from a local directory. 
## The directory should contain the index file and a metadata file that contains the mapping between the document ids and the original documents. 
new_db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization = True)
docs = new_db.similarity_search(query)
docs

[Document(id='5866c05b-3306-438f-9a5b-d5acd6d4578d', metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='23165ef5-9f45-45d8-ad1c-6565c16ead4c', metadata={'source': 'speech.txt'}, page_content='We have borne with their present government through all these bitter months because of that friendship—exercising a pat